# Songo Stockfish - Multi Colab Minimal

Flux minimal:
1. Monter Drive
2. Préparer / mettre à jour le projet (script centralisé)
3. Générer configs actives par identité Colab (script centralisé)
4. Audit stockage (aucune purge)
5. Lancer le pipeline continu dataset (generate + build, sans auto-train)
6. Fusionner les datasets builds des colabs
7. Déclencher train/eval manuellement
8. Déclencher le benchmatch manuellement (cellule séparée)
9. Lancer un tournoi live entre tous les modèles (optionnel)

## 1) Monter Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Setup projet + workspace par identité Google Drive

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

GIT_REPO_URL = 'https://github.com/GlennEriss/songo-model-stockfish-for-google-collab.git'
GIT_BRANCH = 'main'
REPO_NAME = 'songo-model-stockfish-for-google-collab'
DRIVE_PROJECT_NAME = 'songo-stockfish'
# Change cette valeur sur chaque Colab: colab_1, colab_2, colab_3, ...
# Si vide, le script tentera une detection auto Drive.
COLAB_IDENTITY = 'colab_1'

WORKTREE = Path(f'/content/{REPO_NAME}')
PYTHON_BIN = sys.executable or 'python3'

# On clone juste assez pour avoir accès aux scripts centralisés.
if not (WORKTREE / '.git').exists():
    if WORKTREE.exists():
        shutil.rmtree(WORKTREE)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, str(WORKTREE)], check=True)

step_script = WORKTREE / 'scripts' / 'colab' / 'notebook_step.py'
summary_path = Path('/tmp/songo_bootstrap_summary.json')
subprocess.run(
    [
        PYTHON_BIN,
        '-u',
        str(step_script),
        'bootstrap',
        '--git-repo-url',
        GIT_REPO_URL,
        '--git-branch',
        GIT_BRANCH,
        '--worktree',
        str(WORKTREE),
        '--drive-project-name',
        DRIVE_PROJECT_NAME,
        '--colab-identity',
        COLAB_IDENTITY,
        '--python-bin',
        PYTHON_BIN,
        '--summary-path',
        str(summary_path),
    ],
    check=True,
)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
DRIVE_ROOT = Path(summary['drive_root'])
DRIVE_IDENTITY_KEY = str(summary.get('drive_identity_key', '')).strip() or 'unknown_drive_identity'
DRIVE_WORKSPACE_ROOT = Path(summary['drive_workspace_root'])

# Propagation kernel notebook pour les cellules suivantes.
env_updates = {
    'SONGO_DRIVE_ROOT': str(DRIVE_ROOT),
    'SONGO_ENFORCE_DRIVE_ROOT_WRITES': '1',
    'SONGO_DRIVE_IDENTITY_KEY': DRIVE_IDENTITY_KEY,
    'SONGO_DRIVE_WORKSPACE_ROOT': str(DRIVE_WORKSPACE_ROOT),
    'SONGO_WORKTREE': str(WORKTREE),
    'SONGO_PYTHON_BIN': PYTHON_BIN,
}
for _k, _v in env_updates.items():
    os.environ[_k] = _v

print('DRIVE_ROOT           =', DRIVE_ROOT)
print('COLAB_IDENTITY       =', COLAB_IDENTITY or '<auto>')
print('DRIVE_IDENTITY_KEY   =', DRIVE_IDENTITY_KEY)
print('DRIVE_WORKSPACE_ROOT =', DRIVE_WORKSPACE_ROOT)
print('WORKTREE             =', WORKTREE)

## 3) Générer configs actives minimalistes

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        '-u',
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'generate-configs',
        '--worktree',
        WORKTREE,
    ],
    check=True,
)

## 4) Audit stockage (aucune purge)

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        '-u',
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'audit-storage',
    ],
    check=True,
)

## 5) Pipeline continu dataset (sans auto-train)

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
LOG_PATH = Path('/content/songo_streaming_pipeline.log')
cmd = [
    PYTHON_BIN,
    '-u',
    f'{WORKTREE}/scripts/colab/notebook_step.py',
    'streaming-pipeline',
    '--worktree',
    WORKTREE,
    '--disable-auto-train',
    '--heartbeat-seconds',
    '30',
    '--poll-seconds',
    '20',
]

print('Streaming pipeline log file =', LOG_PATH)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
existing_size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0
with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log_file:
    log_file.write('\n=== START streaming-pipeline ===\n')
    log_file.flush()
    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

cursor = existing_size
while True:
    if LOG_PATH.exists():
        with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
            reader.seek(cursor)
            chunk = reader.read()
            if chunk:
                print(chunk, end='')
            cursor = reader.tell()
    rc = proc.poll()
    if rc is not None:
        if LOG_PATH.exists():
            with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
                reader.seek(cursor)
                tail = reader.read()
                if tail:
                    print(tail, end='')
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
        break
    time.sleep(2.0)

## 6) Fusion globale des datasets builds Colab

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
LOG_PATH = Path('/content/songo_merge_built_datasets.log')
NOTEBOOK_STEP = f'{WORKTREE}/scripts/colab/notebook_step.py'
MERGE_SCRIPT = f'{WORKTREE}/scripts/colab/run_merge_built_datasets.py'

help_proc = subprocess.run(
    [PYTHON_BIN, NOTEBOOK_STEP, '-h'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
help_text = help_proc.stdout or ''
has_merge_subcommand = ('merge-built-datasets' in help_text)

if has_merge_subcommand:
    cmd = [
        PYTHON_BIN,
        '-u',
        NOTEBOOK_STEP,
        'merge-built-datasets',
        '--worktree',
        WORKTREE,
        '--heartbeat-seconds',
        '30',
    ]
elif Path(MERGE_SCRIPT).exists():
    cmd = [
        PYTHON_BIN,
        '-u',
        MERGE_SCRIPT,
        '--worktree',
        WORKTREE,
        '--heartbeat-seconds',
        '30',
    ]
else:
    raise RuntimeError(
        'merge-built-datasets indisponible dans ce repo. '
        'Mets le repo a jour (cellule 2) puis relance.'
    )

print('Merge built datasets log file =', LOG_PATH)
print('Command =', cmd)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
existing_size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0
with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log_file:
    log_file.write('\n=== START merge-built-datasets ===\n')
    log_file.flush()
    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

cursor = existing_size
while True:
    if LOG_PATH.exists():
        with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
            reader.seek(cursor)
            chunk = reader.read()
            if chunk:
                print(chunk, end='')
            cursor = reader.tell()
    rc = proc.poll()
    if rc is not None:
        if LOG_PATH.exists():
            with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
                reader.seek(cursor)
                tail = reader.read()
                if tail:
                    print(tail, end='')
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
        break
    time.sleep(2.0)

## 7) Train -> Eval (manuel, sans benchmatch)

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
LOG_PATH = Path('/content/songo_train_eval.log')
cmd = [
    PYTHON_BIN,
    '-u',
    f'{WORKTREE}/scripts/colab/notebook_step.py',
    'run-job',
    'train-eval',
    '--worktree',
    WORKTREE,
    '--heartbeat-seconds',
    '30',
]

print('Train/Eval log file =', LOG_PATH)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
existing_size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0
with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log_file:
    log_file.write('\n=== START train-eval ===\n')
    log_file.flush()
    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

cursor = existing_size
while True:
    if LOG_PATH.exists():
        with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
            reader.seek(cursor)
            chunk = reader.read()
            if chunk:
                print(chunk, end='')
            cursor = reader.tell()
    rc = proc.poll()
    if rc is not None:
        if LOG_PATH.exists():
            with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
                reader.seek(cursor)
                tail = reader.read()
                if tail:
                    print(tail, end='')
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
        break
    time.sleep(2.0)

## 8) Benchmatch manuel (après train+eval)

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
LOG_PATH = Path('/content/songo_benchmark.log')
cmd = [
    PYTHON_BIN,
    '-u',
    f'{WORKTREE}/scripts/colab/notebook_step.py',
    'run-job',
    'benchmark',
    '--worktree',
    WORKTREE,
    '--heartbeat-seconds',
    '30',
]

print('Benchmark log file =', LOG_PATH)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
existing_size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0
with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log_file:
    log_file.write('\n=== START benchmark ===\n')
    log_file.flush()
    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

cursor = existing_size
while True:
    if LOG_PATH.exists():
        with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
            reader.seek(cursor)
            chunk = reader.read()
            if chunk:
                print(chunk, end='')
            cursor = reader.tell()
    rc = proc.poll()
    if rc is not None:
        if LOG_PATH.exists():
            with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
                reader.seek(cursor)
                tail = reader.read()
                if tail:
                    print(tail, end='')
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
        break
    time.sleep(2.0)

## 9) Tournoi des modèles (10 matchs par paire, live)

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
LOG_PATH = Path('/content/songo_model_tournament.log')
cmd = [
    PYTHON_BIN,
    '-u',
    f'{WORKTREE}/scripts/colab/notebook_step.py',
    'model-tournament',
    '--worktree',
    WORKTREE,
    '--games-per-pair',
    '10',
    '--max-moves',
    '400',
]

print('Model tournament log file =', LOG_PATH)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
existing_size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0
with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log_file:
    log_file.write('\n=== START model-tournament ===\n')
    log_file.flush()
    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

cursor = existing_size
while True:
    if LOG_PATH.exists():
        with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
            reader.seek(cursor)
            chunk = reader.read()
            if chunk:
                print(chunk, end='')
            cursor = reader.tell()
    rc = proc.poll()
    if rc is not None:
        if LOG_PATH.exists():
            with LOG_PATH.open('r', encoding='utf-8', errors='replace') as reader:
                reader.seek(cursor)
                tail = reader.read()
                if tail:
                    print(tail, end='')
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
        break
    time.sleep(2.0)